# Budujemy własny leaderboard

## Cel: Zrozumiesz, jak zamienić subiektywne odczucia na precyzyjny system binarnych testów, który pozwoli Ci bezpiecznie rozwijać i kalibrować produkt.

### Czego się nauczysz:
1. Dlaczego gotowe frameworki do ewaluacji są często bezużyteczne w produkcji
2. Jak odróżnić metryki próżności od metryk operacyjnych
3. Czego uczy nas eksperyment z pelikanem na rowerze o testowaniu inteligencji
4. Jak projektować testy binarne zamiast abstrakcyjnych skal punktowych
5. Jak analizować surowe logi konwersacji pod kątem twardych danych
6. Jak stworzyć hierarchię testów od pojedynczego sprawdzenia do pełnego rankingu
7. Jak napisać swój pierwszy inżynierski test logiczny w Pythonie

W tej lekcji zajmiemy się budowaniem własnego rankingu (leaderboardu). To kluczowy moment, w którym przestajesz polegać na opiniach z LinkedIn/Twitter/X, a zaczynasz polegać na własnych danych.

Zanim jednak przejdziemy do pisania kodu i projektowania testów, muszę Cię ostrzec. Bardzo łatwo wpaść tutaj w pułapkę "szybkich rozwiązań". W świecie AI pokusa pójścia na skróty jest ogromna, ale w inżynierii Software 3.0 skróty te zazwyczaj prowadzą na manowce.

Przeanalizujmy to, zanim zaczniesz cokolwiek instalować.


## ⚠️ Pułapka "gotowych rozwiązań"

W lekcji 5.1 ustaliliśmy pewną prawdę: publiczne rankingi zawierają sporo szumu. Naturalnym odruchem osoby technicznej jest więc poszukiwanie narzędzi, które pozwolą zbudować własny ranking i zrobią to za nas.

Mechanizm jest ten sam, co zawsze. Chcesz pamięci dla agenta? Instalujesz gotową bazę wektorową. Chcesz narzędzi? Bierzesz gotowy framework agentowy. Chcesz ewaluacji? Wpisujesz w Google "LLM Evaluation framework" i trafiasz na narzędzia typu [RAGAS](https://docs.ragas.io/en/stable/), [DeepEval](https://deepeval.com) czy inne kombajny.

Obiecują one, że po podpięciu API magicznie otrzymasz piękny dashboard z wynikami:
- Faithfulness (Wierność): 0.78
- Answer Relevancy (Trafność): 0.82
- Context Precision (Precyzja kontekstu): 0.65
- Harmfulness (Szkodliwość): 0.12

![](../images/dashboard.jpg)

Brzmi profesjonalnie? Owszem.
Czy jest użyteczne w produkcji? **Absolutnie nie.**



### Dlaczego to może być problemem?

Problem z tymi narzędziami jest taki sam, jak z każdym wysokopoziomowym abstraktem w AI – tracisz kontrolę i zrozumienie.

Wyobraź sobie, że Twój system produkcyjny (np. chatbot obsługi klienta) dostał ocenę 3.5/5 w kategorii "Helpfulness". Co robisz z tą informacją?

1. Czy model był niepomocny, bo był nieuprzejmy?
2. Czy może podał złą instrukcję?
3. A może po prostu odmówił odpowiedzi ze względów bezpieczeństwa (co w sumie jest pożądane)?

Nie wiesz tego. Masz liczbę, która nic nie znaczy. To są tzw. *Vanity Metrics* (metryki próżności). Wyglądają ładnie na slajdach dla zarządu ("Podnieśliśmy *Faithfulness* o 0,2 punktu. System działa, morale rośnie — premia też mogłaby 😇"), ale nie dają Ci żadnej inżynierskiej dźwigni do naprawy błędów.

Co gorsza, te metryki są abstrakcyjne. Ewaluacji nie da się "zlecić" zewnętrznej bibliotece, bo nikt (często nawet twórcy tych narzędzi) nie wie do końca, jak model "sędzia" definiuje słowo "Faithfulness" w danym momencie. To, co dla generycznego frameworka jest "Harmful" (szkodliwe), dla Twojego systemu medycznego może być krytycznym ostrzeżeniem o interakcji leków. I odwrotnie – coś, co framework uzna za błąd, dla Ciebie może być "ficzerem".

Jeśli Twoja miara jest niestabilna i niejasna, a mierzysz nią niedeterministyczny model, to otrzymujesz chaos do kwadratu. Chcesz?

## Przejście na konkret (Software 3.0)

W inżynierii Software 3.0 musimy zejść na ziemię. Nie interesuje nas abstrakcyjna "Trafność" liczona w zmiennoprzecinkowych odcieniach szarości. Interesuje nas konkret.

Twoim celem jest zamiana "odcieni szarości" (float) na twarde fakty (boolean). Tylko wtedy wiesz, co naprawiać.

Zobacz różnicę:

| ❌ Źle (Generic Metrics) | ✅ Dobrze (Application Specific Metrics) |
| :--- | :--- |
| Faithfulness: Low | **Model zhalucynował datę spotkania** (True/False) |
| Tone: 2/5 | **Model użył wulgaryzmu** (True/False) |
| Helpfulness: Medium | **Model odesłał do nieistniejącego produktu** (True/False) |
| Brevity: High | **Odpowiedź zawiera mniej niż 50 słów** (True/False) |

Widzisz zmianę podejścia?

W kolumnie po prawej stronie nie ma miejsca na domysły. Jeśli świeci się na czerwono test "Model zhalucynował datę", to wiesz dokładnie, gdzie szukać błędu – prawdopodobnie w warstwie RAG (złe dane wejściowe) lub w formacie daty w promptcie systemowym.

To jest fundamentalna zasada budowania własnego leaderboardu: mierzysz to, na czym zależy Twojemu biznesowi, a nie to, co łatwo zmierzyć gotową biblioteką.

## Czy to oznacza, że wszystkie platformy są do kosza?

Nie chciałbym, żebyś wyciągnął stąd wniosek, że cała branża ewaluacji to oszustwo. Spójrzmy na to przez pryzmat inżynierskiego zdrowego rozsądku.

Aby zbudować sensowny system oceniania, musisz najpierw precyzyjnie zdefiniować, na czym Ci zależy w Twoim konkretnym przypadku biznesowym. Jak przekonasz się w dalszej części tej lekcji, to wcale nie jest łatwe zadanie. Wymaga głębokiego zrozumienia domeny, rozmów z klientem i wielu iteracji.

Skoro Tobie – osobie, która tworzy ten system – trudno jest zdefiniować kryteria sukcesu, to jakim cudem zewnętrzna firma (np. twórcy platformy zza oceanu) miałaby to zrobić za Ciebie "przy okazji"? Oni nie znają Twojego biznesu, Twoich danych ani Twoich użytkowników.

Dlatego właśnie te narzędzia uciekają w abstrakcję. Twórcy platformy nie są złośliwi – oni po prostu nie mają innego wyjścia. Muszą zaproponować miary tak ogólne, by pasowały do chatbota prawniczego, generatora wierszy i asystenta kodowania jednocześnie.

A jak coś jest do wszystkiego... no właśnie.

Moja rada jest więc prosta: jeśli bardzo chcesz wdrożyć gotowe kombajny ewaluacyjne – droga wolna. Ale miej świadomość, że patrząc na ten piękny, kolorowy dashboard, prawdopodobnie nadal nie wiesz, co tak naprawdę dzieje się w trzewiach Twojego systemu. Wykres rośnie, słupki są zielone, a użytkownik nadal dostaje błędne odpowiedzi. Nie daj się uśpić fałszywym poczuciem bezpieczeństwa.

## Leaderboard jako sztuka (i inżynieria sprytu) 

Skoro wiemy już, czego NIE robić, zastanówmy się, jak robić to dobrze. Budowanie własnego leaderboardu wymaga zmiany myślenia. To jest coś więcej niż suchy algorytm (napisanie pętli `for` i `assert`). To jest praca twórcza.

Być może myślisz teraz: "Zaraz, nie jestem artystą. Chcę mierzyć twarde dane".

Rozumiem tę obiekcję. Ale w świecie Software 3.0, gdzie systemy są niedeterministyczne, "twarde dane" często zdobywa się sposobem. Istnieje świetna książka Douglasa Hubbarda *"How to Measure Anything"*, która stawia tezę: wszystko da się zmierzyć, jeśli tylko znajdziesz odpowiedni pośrednik (proxy).

Hubbard podaje przykład z teatru. Jak zmierzyć jakość sztuki?
- Liczba sprzedanych biletów? To miara marketingu, nie jakości.
- Opinie krytyków? Zbyt subiektywne i nieliczne.

Okazuje się, że istnieje mierzalny, fizyczny wskaźnik, który idealnie koreluje z zachwytem widowni: **czas trwania oklasków po spektaklu**. Stopery w dłoń i masz twardą daną określającą "niemierzalne" emocje tłumu.

W AI musisz być właśnie takim inżynierem ze stoperem. Trzeba znaleźć sprytny sposób, by złapać model na błędzie, którego nie da się wykryć prostym `if string == string`.

### Case study: Pelikan na rowerze

Podzielę się z Tobą ciekawym podejściem, w którym Simon Willison — twórca benchmarku znanego jako "[Pelican Arena Leaderboard](https://pelicans.borg.games)"

Zadanie dla modelu jest pozornie proste:
> *"Wygeneruj kod SVG przedstawiający pelikana jadącego na rowerze."*

Na pierwszy rzut oka brzmi to jak żart. Co ma wspólnego rysowanie ptaków (swoją drogą, czy możesz sobie teraz przypomnieć, jak wygląda rower, i jeszcze wstawić tam pelikana?) z oceną inteligencji systemu biznesowego? Okazuje się, że to jeden z ciekawych i wartościowych testów ogólnej inteligencji (General Reasoning).

Rozbijmy to na czynniki pierwsze. 

Aby model zaliczył ten test, musi wykonać gigantyczną pracę intelektualną:

1.  **Kodowanie (SVG):** Model nie rysuje pikseli "pędzlem". Model pisze kod XML. Musi znać składnię SVG, rozumieć układ współrzędnych, [krzywe Béziera](https://pl.wikipedia.org/wiki/Krzywa_Béziera) i figury geometryczne.
2.  **Wiedza o świecie:** Musi wiedzieć, jak wygląda pelikan (wielki dziób, krępa sylwetka) i jak wygląda rower (dwa koła, rama, kierownica).
3.  **Rozumowanie przestrzenne (Spatial Reasoning):** To jest klucz. Model musi nie tylko wygenerować rower i pelikana, ale musi je **połączyć**. Pelikan musi *siedzieć* na siodełku, nogi (płetwy?) muszą dotykać pedałów, a dziób nie może przenikać przez kierownicę. To wymaga wyobraźni przestrzennej realizowanej za pomocą kodu.
4.  **Brak w danych treningowych:** W internecie są miliony zdjęć pelikanów i miliony zdjęć rowerów. Ale kod SVG przedstawiający pelikana na rowerze? To rzadkość. Model nie może tego "ściągnąć" z pamięci. Musi to **wymyślić**.


Brzmi ciekawe?

No to zobacz, jakie były wyniki:

### Nowa (Amazon)
![](../images/pelican_1.jpg)

Początki były trudne. Swoją drogą, model Nova (od Amazona) w naszych testach również wypadał bardzo słabo i nie radził sobie dobrze ze structured output, więc to się zgadza. Trudno tam dostrzec jednocześnie rower i pelikana.
Testujemy dalej.

### LLaMa 3 (i to ta największa - 405B)

![](../images/pelican_2.jpg)

Wyszła pewna abstrakcja, ale tutaj można dodać wprost, że LLaMA 3 dość słabo radzi sobie z kodowaniem. Ze structured output jest już lepiej, ale nadal bywa dość kapryśny. W LLaMA 4 wsparcie dla structured output jest już lepsze w porównaniu z wersją 3.


### Gemini, Phi, Qwen

![](../images/pelican_3.jpg)

Dość ciekawie to wygląda, ale model Qwen (ten chiński od Alibaby) zaczyna już wyglądać podobnie do roweru i widać, że ktoś na nim jedzie — niekoniecznie pelikan. I tak jak się domyślasz, Qwen już coś potrafi w zakresie kodowania (są też dedykowane modele Qwen do kodowania, ale to już inna linia).

Trochę przyspieszając historię, zobaczmy ciekawsze przypadki.


### Grok , Kimi (Chiśnki), Gemini

![](../images/pelican_4.jpg)
Tutaj już widać bardzo duży postęp. Na pewno jest rower i jakiś ptak, a czasem nawet można dostrzec podobieństwo do pelikana. Co ciekawe, np. Gemini 2.5 Pro wygenerował dość interesujący kształt roweru oraz pelikana — nie są to zwykłe figury (owale itp.), lecz coś bardziej kreatywnego. Wygląda to całkiem dobrze, można się czepiać szczegółów, ale postęp względem wcześniejszych wyników jest bardzo zauważalny.



### GPT-5, o3-pro, Claude 4 Sonnet

![](../images/pelican_5.jpg)

Tutaj wygląda to już jak dzieło sztuki. W tym przypadku GPT-5 naprawdę wykonał bardzo dobrą robotę, zgadza się? Ale też inne modele są ciekawe. 

## Czego uczy nas ten "dziwny" test?

Możesz zapytać: *"Po co mi to wiedzieć, skoro nie buduję generatora obrazków SVG?"*

Odpowiedź jest kluczowa dla zrozumienia inżynierii ewaluacji. Żadna z firm (OpenAI, Anthropic, Meta) nie optymalizowała swoich modeli pod kątem rysowania ptaków na rowerach. To zadanie jest tak niszowe, że nie istnieje w standardowych zestawach treningowych "fine-tuning".

Aby model mógł to narysować, musi wykazać się **ogólną inteligencją**. Musi połączyć w czasie rzeczywistym trzy kompetencje:
1.  **Umiejętność kodowania (Syntax):** Poprawna składnia XML/SVG.
2.  **Wiedzę o świecie (Knowledge):** Jak wygląda pelikan, a jak rower.
3.  **Logikę przestrzenną (Reasoning):** Relacja obiektów (pelikan musi *siedzieć* na siodełku, a nie lewitować obok).

Jeśli dajemy modelowi zadanie, którego nie mógł "wykuć na pamięć", i mierzymy wynik, który łatwo zweryfikować (obrazek), otrzymujemy potężne narzędzie diagnostyczne.

### Korelacja z rzeczywistością

Co najważniejsze – wyniki tego "śmiesznego" leaderboardu idealnie korelują z naszą inżynierską codziennością. Modele, które trafiły do Top 5 w teście pelikana (Claude 3.5 Sonnet, GPT-4o/5, o3), to dokładnie te same modele, których my i większość branży używamy na co dzień do poważnych zadań programistycznych.

To prowadzi nas do najważniejszego wniosku: **sprytna heurystyka (proxy) bywa lepsza niż skomplikowany framework.**

Jeśli testujesz model do generowania SQL, nie pytaj go o trywialne `SELECT * FROM users`. Zapytaj go o skomplikowane zapytanie analityczne na bazie danych o strukturze, którą wymyślasz w tym samym prompcie (np. baza danych hodowli jedwabników na Marsie). Wtedy sprawdzasz logikę i umiejętność adaptacji, a nie pamięć.


## Case Study: AI Tutor w boju (Vladimir uczy się hiszpańskiego) 🇪🇸

Zostawmy na chwilę ptaki i rowery. Wróćmy do naszego projektu **AI Tutora** (z modułu 2).

Naszym celem nie jest tutaj wygenerowanie ładnego obrazka, ale przeprowadzenie użytkownika przez proces *Onboardingu* i zbudowanie spersonalizowanego planu nauki.

Zobacz, jak wygląda "pelikan" w naszym systemie. To nie jest jeden prompt i jedna odpowiedź. To jest wieloetapowa konwersacja, która w tle buduje skomplikowany stan (State).

Oto zapis rzeczywistej sesji z użytkownikiem (Vladimir), który chce nauczyć się hiszpańskiego.

### Dialog (Warstwa widoczna dla użytkownika)

> **Aoki (AI):** Cześć! Jestem Aoki... Jak masz na imię i jakiego języka chciałbyś się nauczyć?
> **Vladimir:** Mam na imię Vladimir i chcę nauczyć się hiszpańskiego.
> (...)
> **Aoki:** Jak oceniasz swój obecny poziom?
> **Vladimir:** Myślę, że ok. A1 lub A2.
> (...)
> **Aoki:** A jakie konkretne kryteria byś chciał, aby pokazały, że udało ci się to osiągnąć?
> **Vladimir:** np. będę w stanie zamówić jedzenie w Restauracji (...) lub w taksówce jadąc z lotniska.

Z punktu widzenia zwykłego chatu – ot, miła rozmowa. Ale z punktu widzenia inżyniera Software 3.0, to, co dzieje się pod spodem, również jest kluczowe. Model musi "usłyszeć" te informacje i zaktualizować strukturę danych.

### Stan Systemu (Warstwa, którą oceniamy)

To jest nasz "Kod SVG". Jeśli model prowadzi miłą rozmowę, ale na koniec ten JSON jest pusty lub błędny – **test niezaliczony**.

```json
{
  "student_profile": {
    "name": "Vladimir",
    "target_language": "Hiszpański",
    "current_level": "A1",
    "motivation": "podróże",
    "success_criteria": [
      "Będę w stanie zamówić jedzenie w restauracji",
      "Będę mógł porozmawiać w taksówce jadąc z lotniska"
    ]
  },
  "learning_plan": {
    "weekly_hours": 5,
    "milestones": {
      "1_month": "A2+ pewna konwersacja",
      "6_months": "B1 w konwersacji podczas podróży"
    }
  }
}
```

### Projektowanie "Pelikanów" dla Tutora

Widzisz analogię?
W teście pelikana sprawdzaliśmy: *Czy dziób jest połączony z głową?*
W teście Tutora sprawdzamy: *Czy kryteria sukcesu (restauracja, taksówka) trafiły do pola `success_criteria` w JSONie?*

To pozwala nam zdefiniować konkretne testy logiczne:

1.  **Ekstrakcja Faktów (Extraction Logic)**
    *   ✅ *Check:* Czy po zdaniu "Myślę, że ok. A1 lub A2", pole `current_level` zostało ustawione na `A1` (wersja bezpieczna) lub `A1/A2`?
    *   ✅ *Check:* Czy po informacji "6 miesięcy", pole `deadline` zawiera poprawną datę lub okres?

2.  **Prowadzenie Procesu (Workflow Logic)**
    *   ✅ *Check:* Czy model zadał pytanie o *cel nauki*, zanim zapytał o *harmonogram*? (Wymagana kolejność w naszej maszynie stanów).
    *   ✅ *Check:* Czy po zatwierdzeniu planu ("Brzmi super!"), flaga `onboarding_complete` zmieniła się na `True`?

3.  **Halucynacje i Bezpieczeństwo**
    *   ✅ *Check:* Czy w wygenerowanym planie tygodniowym (Monday, Wednesday, Friday) liczba godzin sumuje się do zadeklarowanych 5 godzin? (Modele uwielbiają mylić się w prostej arytmetyce przy generowaniu planów).

Gdybyśmy użyli gotowego narzędzia do ewaluacji i zmierzyli "Answer Relevancy", ta rozmowa dostałaby pewnie 0.95. Wszyscy są mili, kontekst jest zachowany. Ale jeśli model wpisałby "Włoski" zamiast "Hiszpański" w JSONie, system jest **bezużyteczny**.


## Kontrakt jakości

Zadajmy sobie teraz fundamentalne pytanie: **jak w ogóle do tego podejść?**

Brutalna prawda jest taka, że w inżynierii LLM nie ma jeszcze ustandaryzowanych "dobrych praktyk" testowania systemów agentowych. Ta wiedza dopiero dojrzewa. Nie znajdziesz tego w podręcznikach do inżynierii oprogramowania z zeszłej dekady.

W DataWorkshop Lab próbowaliśmy gryźć ten temat na różne sposoby. Testowaliśmy metryki statystyczne, testowaliśmy "czucie" (*vibes*), testowaliśmy proste asercje.

Wnioski?
1.  Metryki statystyczne są ślepe na kontekst
2.  "Vibes" są nieskalowalne
3.  Proste asercje są zbyt kruche

To, co finalnie udało nam się wypracować i co stało się naszym wewnętrznym standardem, to koncepcja **Leaderboard Card**.


Nasz kontrakt nie jest płaską listą życzeń. Ma swoją ścisłą hierarchię, która pozwala zarządzać chaosem (bo jak wiesz, w AI często naprawienie jednej rzeczy psuje drugą).


Zanim pokażę Ci pełną tabelę ze specyfikacją (w kolejnej lekcji), musisz zrozumieć trzy fundamentalne zasady, na których ją oparliśmy:

### 1. Zasada Binarności (The Binary Rule)
W tym kontrakcie nie ma miejsca na oceny w skali 1-5 ("było całkiem nieźle").
Interesuje nas system zero-jedynkowy:
- **1 (Pass):** Kryterium spełnione w 100%.
- **0 (Fail):** Wystąpił błąd, brak odpowiedzi lub odpowiedź wymijająca.

Dlaczego? Bo tylko binarny wynik daje Ci jasną informację zwrotną. Albo masz `current_level` w JSON-ie, albo go nie masz. Nie możesz mieć go "trochę".

### 2. Hierarchia błędów 
Nie wszystkie błędy są sobie równe.
*   🔴 **MUST-HAVE (Critical):** To są "showstoppers". Jeśli model poda uczniowi gotowe rozwiązanie zadania domowego zamiast go naprowadzić – system oblewa test, niezależnie od tego, jak miły był przy tym coach. To jest naruszenie *Prime Directive*.
*   🟡 **SHOULD-HAVE (Quality):** To aspekty wpływające na jakość doświadczenia (UX). Jeśli coach zapomni o podsumowaniu lekcji, to jest błąd, ale nie krytyczny. System jest "Acceptable", ale nie "Diamond".

### 3. Hierarchia Leaderboardu 
Aby zarządzać setkami takich testów, potrzebujemy struktury:

1.  **Check (Atomowy Test):** Najmniejsza jednostka. Zawsze zwraca `True` lub `False`.
    *   *Przykład:* "Czy odpowiedź zawiera emotikony?"
2.  **Grupa (Kategoria):** Zbiór powiązanych Checków.
    *   *Przykład:* "Dydaktyka", "Ochrona przed Prompt Injection".
3.  **Leaderboard (Dashboard):** Widok zarządczy pokazujący wyniki grupami.


Podam Ci przykład

1.  **Atomowy Test (Check):** Najmniejsza jednostka. Musi być binarna (0/1). Albo działa, albo nie.
2.  **Kategoria (Area):** Zbiór testów dotykających jednego tematu (np. "Bezpieczeństwo", "Dydaktyka").
3.  **Priorytety (The Hierarchy of Needs):** To jest klucz do sukcesu. Podzieliliśmy wymagania na trzy koszyki:
    *   🔴 **MUST-HAVE:** Krytyczne błędy ("Showstoppers"). Jeśli to nie działa, system nie wychodzi na produkcję.
    *   🟡 **SHOULD-HAVE:** Ważne dla jakości, ale system jest używalny bez tego (np. UX, styl).
    *   🟢 **NICE-TO-HAVE:** "Wisienki na torcie".
4.  **Formuła (The Score):** Na końcu mamy matematyczny wzór, który automatycznie wylicza "Poziom Jakości" (np. Silver, Gold, Diamond).

### Broń na niejednoznaczność: Przykłady +/-

Zwróć uwagę na jedną kluczową rzecz w powyższym kontrakcie. Przy każdym kryterium nie kończymy na suchej definicji. Zawsze dodajemy sekcję **Przykłady pozytywne** oraz **Przykłady negatywne**.

---
#### PT-001 🔴 MUST-HAVE - Priorytet 1 | **Sesje: Wszystkie**
- **Kryterium**: Czy coach prowadzi rozmowę w języku ojczystym ucznia (podczas onboardingu i wyjaśnień)?
- **Weryfikacja**: Sprawdź język wypowiedzi coacha vs zadeklarowany język ojczysty ucznia
- **Ocena**: 1 = konsekwentny język ojczysty dla instrukcji/wyjaśnień, 0 = błędny język
- **Przykłady pozytywne**:
  1. Uczeń polski → Coach wyjaśnia po polsku "Dzisiaj będziemy ćwiczyć zamawianie w restauracji"
  2. Coach pyta o preferencje językowe przy niejednoznaczności
  3. Instrukcje gramatyczne w języku ojczystym
- **Przykłady negatywne**:
  1. Uczeń polski → Coach wyjaśnia tylko po angielsku
  2. Skomplikowane instrukcje w języku docelowym dla poziomu A1
  3. Mieszanie języków w sposób dezorientujący ucznia
---

Dlaczego to jest tak ważne?

Bo definiowanie reguł w teorii jest łatwe, ale w praktyce (na "żywym organizmie") okazuje się koszmarem niejednoznaczności.

#### 1. Synchronizacja zespołu
Wyobraź sobie sytuację: Wdrażasz model. Tester zgłasza błąd: *"Model użył hiszpańskiego słowa w polskim zdaniu"*. Ty jako inżynier mówisz: *"No tak, ale to było słowo kluczowe, musiał go użyć!"*.

Zaczyna się dyskusja. Biznes mówi: *"No nie, w tej sytuacji to wyglądało źle"*.
Zamiast kłócić się o abstrakcyjne definicje, robicie jedną prostą rzecz: **dodajecie ten konkretny przypadek (edge case) do listy Przykładów Negatywnych w kontrakcie.**

Dzięki temu kontrakt jest **żyjącym dokumentem**. Jeśli pojawia się nowa, sporna sytuacja – dopisujesz jeden punkt w Markdownie. Cały zespół (i biznes, i inżynierowie) natychmiast się synchronizuje. Bez tego mechanizmu utoniecie w interpretacjach słowa "pomocny" czy "poprawny".

#### 2. Instrukcja dla sędziego 
Pamiętaj, kto będzie czytał ten dokument. Nie tylko człowiek, ale przede wszystkim **LLM-Sędzia**, który będzie automatycznie oceniał rozmowy.

Dla modelu te przykłady to nic innego jak technika **Few-Shot Prompting**.
Podając mu konkretne przykłady ("To jest OK", "To jest BŁĄD"), drastycznie zwiększasz precyzję jego oceny. Model przestaje "zgadywać", co autor miał na myśli, a zaczyna dopasowywać wzorce.

To jest piękno formatu Markdown. Jest on czytelny dla człowieka (łatwo edytować na GitHubie) i natywny dla modelu. To najtańszy i najskuteczniejszy sposób na podniesienie jakości Twojego leaderboardu.

### Markdown jako kod

Bardzo ważna uwaga techniczna: **Taka karta musi być napisana w czystym Markdown.**

Dlaczego? Ponieważ ten dokument ma dwóch czytelników:
1.  **Ty:** Musisz rozumieć, czego wymagasz od systemu.
2.  **LLM:** który "przeczyta" ten kontrakt i na jego podstawie oceni rozmowę. Markdown jest idealnym formatem – czytelnym dla ludzi i natywnym dla LLM-ów.

Dlatego my traktujemy te karty jak kod. Trzymamy je w repozytorium (Git/GitHub). Mamy tam wersjonowanie, historię zmian (*Git Blame*) i proces Code Review. Dzięki temu widzimy czarno na białym: *"Wersja v1.2 kontraktu poluzowała wymogi co do formatowania, ale zaostrzyła wymogi co do bezpieczeństwa"*.



### Dlaczego to jest lepsze niż "Score 1-10"?

Wyobraź sobie, że wdrażasz nowy model. Puszczasz testy.

W starym podejściu (generic score) widzisz:
> "Overall Score spadł z 4.8 na 4.6"

Drapiesz się po głowie. Co się stało? Nie wiesz.

W nowym podejściu (binary checks) widzisz:
> "Grupa 'Dydaktyka': Spadek o 15%."
> "Szczegóły: Check 'Czy model podał gotową odpowiedź' zaczął zwracać TRUE w 20% przypadków."

Masz natychmiastową diagnozę: **Nowy model jest zbyt chętny do pomocy i "spojleruje" wyniki.**
Rozwiązanie: Musisz wzmocnić Prompt Systemowy w sekcji "Zasady niepodawania rozwiązań".

### Gdzie chcesz się zmęczyć?

Tu dochodzimy do kluczowej zasady projektowania systemów ewaluacji. Wybór między skalą (1-10) a binarnością (0/1) to tak naprawdę decyzja o tym, **w którym momencie chcesz wykonać pracę intelektualną.**

Masz dwie opcje:
1.  Męczysz się **PRZED** testem (podejście Binarne): Musisz mocno pomyśleć, jak zamienić swoje subiektywne odczucie w twardy warunek logiczny. To trudne. Wymaga precyzji. Ale potem, gdy analizujesz wyniki, masz jasność.
2.  Męczysz się **PO** teście (podejście Skalarne): Definiujesz test szybko ("Oceń jakość 1-5"). Ale potem, gdy patrzysz na wyniki, musisz się "naklęskować", żeby zrozumieć, czym różni się ocena 4 od 3.5 i czy to w ogóle jest błąd krytyczny.

Domyślnie chcemy sprowadzać wszystko do **metryk binarnych**, bo one zdejmują z nas ciężar interpretacji w przyszłości.

Oczywiście, nikt nie zabroni Ci używać skali 1-5. Czasem jest to potrzebne. Ale uważaj na to, bo przesuwasz wtedy suwak na osi niejednoznaczności. Im bardziej złożona metryka (binarna -> 1-5 -> 1-10 -> opis słowny), tym trudniej będzie Ci potem wyciągnąć właściwe wnioski i podjąć decyzję o wdrożeniu na produkcję (Pass/Fail).

Dlatego w inżynierii opłaca się "zainwestować" myślenie na początku, przy definicji testu, żeby potem spać spokojnie, gdy dashboard świeci się na zielono. To jest różnica między zgadywaniem a inżynierią. I dokładnie taki system – oparty na binarnym kontrakcie.

## Anatomia rozmowy: wczytujemy dane testowe

Skoro mamy już nasz kontrakt (teorię), potrzebujemy materiału dowodowego (praktyki).

Pamiętasz nasz przykładowy dialog z modułu 2? 

Tu jest przykładowa rozmowa. Uruchomiłem tę sesję, pozwoliłem systemowi zebrać dane i wyeksportowałem całość do pliku JSON. To jest nasz "pacjent".

Z inżynierskiego punktu widzenia, ewaluacja to nic innego jak funkcja:
`f(conversation_log, internal_state) -> quality_score`

Wczytajmy zatem te dane i zobaczmy, najpierw oczami, z czym będziemy pracować (nie musisz czytać wszystkiego — na początek wystarczy, że poskrolujesz).

Swoją drogą, jeśli chcesz, możesz wygenerować własny dialog (w 2 module, na końcu lekcji 2.5, pokazano, jak uruchomić agenta; potem rozmawiasz i na koniec klikasz `Advanced → Download JSON`), dzięki czemu będziesz mieć swój dialog.

![](../images/ai_tutor_expert.jpg)

### Wczytanie pliku 

In [ ]:
with open("../input/ai_tutor_1.json", "r") as f:
    dialog = f.read()
print(dialog)    

### Wynik

Oto co zobaczymy w konsoli. To jest **kompletny zrzut pamięci** naszego agenta po zakończeniu onboardingu.

```json
{
  "export_metadata": {
    "user_id": "test_user2",
    "export_timestamp": "2025-12-01T10:31:48.064545+00:00",
    "onboarding_complete": true
  },
  "student_profile": {
    "name": "Vladimir",
    "native_language": "Polski",
    "target_language": "Hiszpański",
    "current_level": "A1",
    "motivation": "podróże",
    "goal": "Będę w stanie zamówić jedzenie w restauracji...",
    "target_level": "B2",
    "deadline": "6 miesięcy",
    "success_criteria": [
      "Będę w stanie zamówić jedzenie w restauracji",
      "Będę mógł porozmawiać w taksówce jadąc z lotniska"
    ]
  },
  "conversation_history": [
    {
      "role": "user",
      "content": "Cześć"
    },
    {
      "role": "assistant",
      "content": "Cześć! Jestem Aoki, twoim coachem językowym..."
    },
    // ... (reszta wiadomości)
    {
      "role": "assistant",
      "content": "Znakomicie, Vladimir! Oto zaktualizowany plan nauki..."
    }
  ],
  "full_state": {
    "core": {
      "prime_directives": [
        "NO_HOMEWORK_SOLVING",
        "COMMUNICATION_FIRST"
      ]
    },
    "semantic": {
      "learning_plan": {
        "weekly_hours": 5,
        "milestones": {
          "1_month": "A2+ pewna konwersacja",
          "6_months": "B1 w konwersacji podczas podróży"
        }
      }
    }
  }
}
```


### Co my tu właściwie mamy?

Ten JSON to złoto dla inżyniera ewaluacji. Zwróć uwagę na kluczowe sekcje, które będziemy badać:

1.  **`conversation_history` (Transcript):**
    To jest warstwa widoczna dla użytkownika. Nasz LLM-Sędzia będzie czytał ten tekst, aby ocenić "miękkie" aspekty: ton wypowiedzi, dydaktykę, empatię i czy coach nie halucynował.

2.  **`student_profile` (Extracted Facts):**
    To jest wynik działania "inteligencji" systemu. Tutaj sprawdzamy twarde fakty (Extraction Logic).
    *   Czy imię to Vladimir? (Tak)
    *   Czy poziom to A1? (Tak)
    *   Jeśli tutaj byłoby `name: null` albo `target_language: Włoski`, to system oblałby test, nawet gdyby rozmowa była przemiła.

3.  **`full_state` (Internal Logic):**
    To są "bebechy" maszyny stanów. Widzimy tu `prime_directives` (czy system pamięta swoje zasady?) oraz wygenerowany `learning_plan`. To tutaj sprawdzamy spójność logiczną (np. czy kamienie milowe pasują do deadline'u).

### ☝️ Ważna uwaga: Nie panikuj przy czytaniu!

Nie musisz teraz analizować każdego nawiasu w tym pliku. To normalne, że taki JSON ma setki linii.
Traktuj ten plik jako **Dataset**. W inżynierii Software 3.0 rzadko czytamy te logi "oczami" od deski do deski. Zamiast tego piszemy kod, który wyciąga z nich to, co nas interesuje.

W tej chwili wystarczy, że rozumiesz, że mamy **dwa źródła prawdy**:
1.  To, co zostało powiedziane (`conversation_history`).
2.  To, co zostało zrozumiane i zapisane (`student_profile` / `full_state`).

Naszym zadaniem w kolejnych krokach będzie sprawdzenie, czy te dwa światy są ze sobą zgodne.

## 🧠 Zadanie 5.2.1: Definiujemy własny "check"

To jest najważniejszy moment tej lekcji. Patrzysz na mój dialog. Widzisz, że rozmowa poszła dość gładko. AI Tutor zapytał o imię, zrozumiał cel, wygenerował plan.

Ale tutaj czai się pułapka. To, że model raz zrobił coś dobrze (przypadkiem), nie znaczy, że zrobi to dobrze jutro.
W Software 3.0 obowiązuje zasada: jeśli czegoś nie mierzysz w Leaderboardzie, to dla Ciebie to nie istnieje.

Jeśli podoba Ci się, że AI Tutor zapytał o "termin (deadline)" – musisz stworzyć na to test. Jeśli tego nie zrobisz, przy aktualizacji (czegoś, np. struktury, promptu, modelu i wiele innych czyników) ta cecha może zniknąć, a Ty nawet tego nie zauważysz.

### Twoim zadaniem jest:
Na podstawie przeanalizowanego dialogu (sekcja wyżej), zdefiniuj **jeden konkretny test (Check)** w formacie Markdown.
Wybierz coś, co Twoim zdaniem jest krytyczne dla sukcesu tego produktu.

Skorzystaj z tego szablonu:
```markdown
#### [ID] 🔴 MUST-HAVE (lub 🟡 SHOULD-HAVE)
- **Kryterium**: [Co sprawdzamy?]
- **Weryfikacja**: [Gdzie i jak szukać dowodów w JSON?]
- **Ocena**: 1 = [Warunek sukcesu], 0 = [Warunek porażki]
- **Przykłady pozytywne**:
  1. [Przykład A]
- **Przykłady negatywne**:
  1. [Przykład B]
```

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź
    </summary>
<p>

### Wskazówki:

1.  **Szukaj faktów w JSON:** Zobacz sekcję `student_profile`.  W JSON pojawiło się to w `success_criteria`. To idealny kandydat na test!
2.  **Szukaj logiki procesu:** AI Tutor najpierw zapytał o cel, a dopiero potem przedstawił plan. Czy gdyby zrobił odwrotnie, byłoby OK? Jeśli nie – zrób z tego test(zdefiniuj to jako ważna miara dla Ciebie, jako część kontraktu).
3.  **Szukaj bezpieczeństwa:** Czy AI Tutor nie obiecał czegoś nierealnego (np. "Wyślę ci książkę pocztą")?

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć przykładowe rozwiązanie
    </summary>
<p>

Oto przykład testu opartego na "Kryteriach Sukcesu", który jest kluczowy w tej rozmowie:

#### CN-003 🔴 MUST-HAVE - Priorytet 1 | **Sesje: Onboarding**
- **Kryterium**: Czy coach ustalił konkretne kryteria sukcesu (po czym uczeń pozna, że osiągnął cel)?
- **Weryfikacja**: Sprawdź pole `success_criteria` w `student_profile` oraz ostatnie wypowiedzi użytkownika przed wygenerowaniem planu.
- **Ocena**: 1 = kryteria są konkretne i mierzalne (np. zamówienie jedzenia), 0 = brak kryteriów lub są ogólnikowe ("będę umiał język")
- **Przykłady pozytywne**:
  1. Uczeń: "Chcę zamówić jedzenie w restauracji" → JSON: success_criteria zawiera tę frazę.
  2. Uczeń: "Chcę zdać egzamin B2" → JSON: success_criteria zawiera "Egzamin B2".
- **Przykłady negatywne**:
  1. Coach nie zapytał o kryteria sukcesu.
  2. Uczeń podał kryteria, ale pole w JSON jest puste (`[]`).
  3. Kryterium to "Będę mówił lepiej" (niemierzalne).

</p>
</details>
</p>
</details>

In [ ]:
leaderboard_check = """

HERE

"""

## Dlaczego masz to robić lub kto za to odpowiada? 

Zadanie, które przed Tobą stoi (zdefiniowanie testów), jest nietypowe. Być może myślisz teraz:
*"Zaraz, nie jestem testerem. Od definiowania wymagań jest Biznes / Product Owner / QA. Oni powinni mi dostarczyć ten plik, a ja mam go tylko zaimplementować."*

W idealnym świecie? Tak.
W rzeczywistości? Zapomnij o tym.

### Brutalna prawda o wymaganiach
Większość interesariuszy biznesowych nie jest w stanie dostarczyć Ci binarnego Leaderboardu. Dostaniesz od nich co najwyżej ogólniki:
- "Chcemy system, który działa dobrze"
- "Ma być miły i pomocny"
- "Niech użytkownicy dają mu 10/10 w ankiecie"

Problem w tym, że te miary są "brudne" i opóźnione. "Zadowolenie użytkownika" to wypadkowa setek czynników (UI, szybkość, humor użytkownika tego dnia). Jako inżynier nie masz dźwigni, żeby wprost wpłynąć na "NPS" (*Net Promoter Score*) poprzez zmianę promptu. Potrzebujesz miar, które są blisko "metalu" – blisko modelu.

Skoro Biznes (zwykle) nie potrafi przełożyć "fajności" na "boolean", powstaje próżnia odpowiedzialności.

### Nowa rola: AI QA Architect?
Być może w przyszłości powstanie nowa, dedykowana rola (AI Quality Assurance), która zajmie się tylko tym. Ale dzisiaj, w pionierskich czasach Software 3.0, ta rola spada na Ciebie.

I tu dobra wiadomość: **To Twoja supermoc, a nie przykry obowiązek.**

Podejście "Full Stack AI Engineer" wymaga wciągnięcia kompetencji produktowych. Dlaczego to Ci się opłaca?
Bo ten Leaderboard to nie jest tylko narzędzie do oceny. To jest specyfikacja architektury.

Jeśli wiesz, że w Leaderboardzie jest test *"Czy system pamięta imię z poprzedniej sesji"*, to wiesz, że musisz zaprojektować strukturę, w której na pewo musi być "`name`". Bez Leaderboardu strzelasz w ciemno.

### Efekt lawiny

Opowiem Ci historię z naszego labu, która pewnie wyda Ci się znajoma.

Zanim wdrożyliśmy Leaderboardy, każda sesja demo z klientem wyglądała jak lawina.
- Poniedziałek: "Model jest zbyt sztywny, poluzujcie go!"
- Wtorek (po poluzowaniu): "Model gada głupoty, niech będzie bardziej profesjonalny!"
- Środa: "Dlaczego on pyta o to, a nie o tamto?"
- ...
- ...
- 🤯

To był chaos. Bieganie od ściany do ściany. Bez twardych punktów odniesienia każda subiektywna opinia stawała się rozkazem do zmiany architektury.

Wprowadzenie **Must-Have / Should-Have** zmieniło wszystko.

Zadziałała strategia **"zamrażania"**:
1.  Najpierw dowieźliśmy **Must-Have** (np. "Nie podaje gotowych rozwiązań"). Gdy to działało na zielono – **zamroziliśmy to**.
2.  Dopiero wtedy zajęliśmy się **Should-Have** (np. "jest miły").
3.  Jeśli klient zgłaszał uwagę: "Jest mało zabawny", sprawdzaliśmy: "Czy 'Bycie zabawnym' jest w Must-Have?". Nie? To trafia do backlogu, a my nie psujemy działającego systemu.

Dzięki temu w ogóle dowieźliśmy produkt. Z perspektywy czasu rozumiem, że w tej konfiguracji była to jedyna słuszna opcja (żadnej innej nie udało się wymyślić). Co ciekawe, dzięki takiemu podejściu można też lepiej zarządzać zespołem we właściwy sposób, bo każdy dostaje swoją miarę (np. jedna grupa na konkretną osobę lub grupę osób) i zwykle wtedy nie przeszkadzają nawzajem oraz panuje porządek. Wiadomo, kto co ma zrobić.


Nawet jeśli to nie Ty finalnie wpiszesz te literki do pliku Markdown (może zrobisz to z kolegą z biznesu), to Ty musisz znać ten Leaderboard na pamięć. On jest Twoją tarczą przed chaosem i mapą dla Twojego kodu.

Dlatego w tym zadaniu proszę Cię o zdefiniowanie chociaż jednej miary. Aby poczuć, jak to robi się – i jak potężne – jest zamienienie "chcę fajnego bota" na "funkcja zwraca True".

## Podsumowanie

Brutalna prawda jest taka: *Leaderboardy* to nie opcja, to konieczność.

Jeśli masz zapamiętać z tej lekcji tylko jedną rzecz, niech to będzie myśl, że produkty LLM w Software 3.0 nie są tworzone raz na zawsze — są aparatem do ciągłej kalibracji.

Pomyśl o tym logicznie: bez leaderboardu dalsze kroki inżynierskie nie mają sensu.

1. Brak nawigacji: Nie możesz niczego ulepszać, bo nie wiesz, czy Twoja zmiana (np. przejście na nowszy model) nie popsuła czegoś innego. Leaderboard to Twój mechanizm, który precyzyjnie wskaże: "Tu jest dobrze, ale tam się posypało".
2. Kreatywna inżynieria: Przykład z pelikanem na rowerze pokazuje, że tworzenie testów to praca twórcza, ale nie skomplikowana matematycznie. Trzeba po prostu wypracować nawyk szukania sprytnych sposobów na "złapanie" modelu. Gdyby dało się to zrobić jednym algorytmem, ludzie nie byliby potrzebni.
3. Ty jesteś kapitanem: Kto ma tworzyć te rankingi? Na rynku nie ma jeszcze roli "AI QA", a biznes często nie potrafi przełożyć "oczekiwań" na "kod". My, inżynierowie, byliśmy w pewnym sensie postawieni pod ścianą — musieliśmy wymyślić tę metodę, bo bez niej tonęliśmy w chaosie subiektywnych opinii.

### Co dalej?
W kolejnej lekcji pokażę Ci, dokąd dążymy. Nie chcę Cię teraz przerażać pełną złożonością, ale zobaczysz rozbudowany *Leaderboard Card*. Gdy nabierzesz wprawy (wierzę, że dasz rady), będziesz się z tym rozpędzać znacznie szybciej. Pokażę Ci konkretne przykłady definicji, które pozwolą Ci zbudować w głowie mentalny model kompletnego systemu ewaluacji.

Brzmi jak plan? No to działajmy!

## Przydatne linki

1. [The last six months in LLMs, illustrated by pelicans on bicycles](https://simonwillison.net/2025/Jun/6/six-months-in-llms/) + [youtube](https://www.youtube.com/watch?v=YpY83-kA7Bo)
2. [Pelican Arena Leaderboard](https://pelicans.borg.games)
3. [LLM Leaderboard](https://www.vellum.ai/llm-leaderboard)
4. [Artificial Analysis Music Arena Leaderboard](https://huggingface.co/spaces/ArtificialAnalysis/Music-Arena-Leaderboard)
5. [FrontierMath — benchmarking AI against advanced mathematical research](https://epoch.ai/frontiermath)
6. [FieldWorkArena: Agentic AI Benchmark for Real Field Work Tasks](https://arxiv.org/pdf/2505.19662)
7. [τ-bench  (airline, retail, telco)](https://taubench.com/#leaderboard)